# Build your first neural network

Room310 · Deep learning foundations

## Goal

Build and train a 2 → 8 → 1 network, then inspect all four of its binary predictions.

## Setup

Run this notebook from top to bottom. It is self-contained and uses only synthetic teaching data. No credentials or dataset downloads are needed. Save a copy before editing.

Use a CPU Python environment with PyTorch installed. In Colab, connect to the default CPU runtime. If `import torch` fails, run `%pip install torch` in a separate cell and restart the kernel if asked. For local installation, follow https://pytorch.org/get-started/locally/.

Examples use a fixed seed where relevant; exact floating-point results can vary by environment.

## Steps

### 1. The XOR puzzle

**XOR** means exclusive or: output 1 when two bits differ, otherwise 0. Its four examples are (0,0) → 0, (0,1) → 1, (1,0) → 1, and (1,1) → 0.

Imagine those inputs as the corners of a square. The positive examples sit on opposite corners. No single straight line separates them from the other two corners. A linear model with one output and a threshold cannot solve this pattern.

This is a tiny learning exercise, not a generalization benchmark: we train on all four possible binary inputs. Lesson 6 will measure predictions on genuinely held-out examples.

In [1]:
import torch
from torch import nn

torch.manual_seed(7)
X = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y = torch.tensor([[0.0], [1.0], [1.0], [0.0]])
print("Inputs:", X.shape, "Targets:", y.shape)

Inputs: torch.Size([4, 2]) Targets: torch.Size([4, 1])


**Check your result:** There are four examples with two features each and one binary target per example.

### 2. Two layers and a bend in the middle

`nn.Sequential` passes values through its layers in order. `nn.Linear(2, 8)` makes eight weighted combinations of two inputs, each with its own bias. `nn.Tanh()` bends those values into the range -1 to 1. `nn.Linear(8, 1)` combines the hidden values into one final score.

Without a nonlinear activation between them, stacking these linear layers would still be one linear calculation. The nonlinear hidden layer is what allows a more flexible boundary.

The output is a **logit**, an unrestricted score—not a probability yet. `BCEWithLogitsLoss` combines a sigmoid transformation with binary cross-entropy in a numerically stable calculation. Do **not** add a sigmoid before this loss. For this loss, the targets are floating-point 0s and 1s with the same shape as the logits.

An **optimizer** updates the parameters. Here we use Adam with a learning rate of 0.03. It adjusts update sizes using gradient history; it still depends on gradients from backward().

In [2]:
model = nn.Sequential(
    nn.Linear(2, 8),
    nn.Tanh(),
    nn.Linear(8, 1),
)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

Sequential(
  (0): Linear(in_features=2, out_features=8, bias=True)
  (1): Tanh()
  (2): Linear(in_features=8, out_features=1, bias=True)
)
Trainable parameters: 33


**Check your result:** There are 33 parameters: 2×8 weights + 8 biases in the first layer, then 8×1 weights + 1 bias in the second.

### 3. Train, then inspect each prediction

The loop should now look familiar. `optimizer.zero_grad()` replaces our manual gradient clearing; `optimizer.step()` replaces the manual subtraction. The model starts in training mode, then switches to evaluation mode for inference.

`model.eval()` changes the behavior of certain layers, such as dropout; it does not turn gradient tracking off. That is why we also use `torch.no_grad()`. Our simple Linear/Tanh network has no dropout, but this is a useful habit.

At inference time, sigmoid turns a logit into a value between 0 and 1. We interpret this as the model's estimated probability of class 1 and use a 0.5 threshold. A confident score is not a guarantee that a model is correct.

In [3]:
model.train()
for epoch in range(400):
    optimizer.zero_grad()
    logits = model(X)
    assert logits.shape == y.shape
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch:3d} | loss before update: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    probabilities = torch.sigmoid(model(X))
    predictions = (probabilities >= 0.5).float()
    accuracy = (predictions == y).float().mean().item()
for inputs, probability, predicted, target in zip(X, probabilities, predictions, y):
    print(inputs.tolist(), f"p(1)={probability.item():.3f}",
          "prediction=", int(predicted.item()), "target=", int(target.item()))
print(f"Training accuracy: {accuracy:.0%}")

Epoch   0 | loss before update: 0.7026


Epoch 100 | loss before update: 0.0065


Epoch 200 | loss before update: 0.0016


Epoch 300 | loss before update: 0.0008


[0.0, 0.0] p(1)=0.000 prediction= 0 target= 0
[0.0, 1.0] p(1)=1.000 prediction= 1 target= 1
[1.0, 0.0] p(1)=1.000 prediction= 1 target= 1
[1.0, 1.0] p(1)=0.001 prediction= 0 target= 0
Training accuracy: 100%


**Check your result:** With the supplied seed and settings, expect predictions 0, 1, 1, 0 and 100% training accuracy. Exact probabilities can vary across PyTorch versions. This only proves the tiny training table was learned.

## Checks

You have trained a real, small neural network. Nonlinearity lets it learn XOR; a separate evaluation is needed to judge unseen data.

Compare your output with each check above. Explain unexpected results before moving on.

## Next Steps

### Practice & explain

### Remove the activation

Create a fresh model with just Linear(2, 8) and Linear(8, 1), recreate the optimizer, and train again. Compare the predictions and loss with the original network. Explain why extra linear layers alone cannot solve XOR.

<details><summary>Need a hint?</summary>

When you replace a model, the old optimizer still refers to the old parameters. Create a new optimizer for the new model.

</details>

In [4]:
# Your experiment or explanation goes here.


### Try a smaller hidden layer

Compare 2, 4, and 8 hidden neurons. Reset the seed, model, and optimizer for each experiment. Record parameter count, training loss, and all four predictions; do not claim that every initialization will converge.

<details><summary>Need a hint?</summary>

Change both Linear(2, hidden_size) and Linear(hidden_size, 1). With h hidden neurons, there are 4*h + 1 parameters.

</details>

In [5]:
# Your experiment or explanation goes here.


### References

- [PyTorch · building a neural network](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
- [PyTorch · optimizing model parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)